# Tutorial about fluopy

Here we outline the principle use of fluopy.

In [ ]:
from pprint import pprint

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

import fluopy
import fluopy.analysis as an
import fluopy.blinking as bl
import fluopy.emissions as em
import fluopy.fcs as fcs_p
import fluopy.figure as fi
import fluopy.fluorophores as fl
import fluopy.miscellaneous as mi
import fluopy.prediction as pr
import fluopy.simulation as si
import fluopy.transitions as tr

In [ ]:
fluopy.__version__

For random number generation a Generator is initialized.

In [ ]:
rng = np.random.default_rng(seed=1)

## Modules

Fluopy contains the following modules. For detailed information look up the API documentation.

In [ ]:
print("Fluopy modules:")
print()
pprint([item for item in dir(fluopy) if not item.startswith("_")])

## Introduction

Fluopy helps to run Markov chain simulations for a photophysical system of isolated or communicating fluorophores.

To get started the photophysical system has to be defined and instantiated.

This consists of the following steps:

* Define a fluorophore system that contains fluorophores at positions in space.
* Define a set of possible transitions for the fluorophore system.

A simulation provides a series of transition events that occured in a single realization of the Markov chain.

A simulation is run for a given fluorophore system and transition set with a random number generator initialized as seed.

For a fluorophore system and transition set that fullfill certain requirements a prediction for the simulation outcome can be derived based on computational estimation of a steady-state equilibrium.

Finally, the acquired data is analysed and presented in various formats.

Analysis procedure that focus on experimentally observable transitions are available:

* Time traces
* Fluorescence correlation spectroscopy
* ON/Off time distributions

A simulation with pulsed excitation can be carried out. The analysis resembles a TSCPC experiment.

A typical setup, simulation and analysis procedure for selected photophysical systems is presented in the other tutorials. 

In the following we will present the relevant code blocks.

## Transitions

There are various transition types that connect the photophysical states:

In [ ]:
list(tr.TransitionType.__members__)

Photophysical states are named according to the Enum class `tr.SingleState` and `tr.PairedState`, which link state IDs to photophysical state names.

A single fluorophore can occupy the following photophysical states:

In [ ]:
list(tr.SingleState.__members__)

Two interacting fluorophores can occupy the following (paired) photophysical states:

In [ ]:
list(tr.PairedState.__members__)

A specific transition of a certain type from state 1 to state 2 is defined as:

In [ ]:
transition = tr.Transition(
    transition_type=tr.TransitionType.EXCITATION, rate=1e6, fluorophore_ids=[1]
)
pprint(transition)

A photophysical system is made up of one or more fluorophores or fluorophore pairs with multiple transitions:

In [ ]:
transition_1 = tr.Transition(
    tr.TransitionType.EXCITATION, rate=1e6, fluorophore_ids=[0]
)
transition_2 = tr.Transition(
    tr.TransitionType.FLUORESCENT_EMISSION, rate=1e9, fluorophore_ids=[0]
)
transitions = {
    "cy5_dna": [transition_1, transition_2],
}
pprint(transitions)

## The fluorophore system

A photophysical system is defined by one or more fluorophores that occupy electronic states and go through transitions from one state to another.

A number of input parameters are kept in fluopy.fluo_data as FluorophoreData instances. These are automatically inserted if the name matches.

In [ ]:
fluorophore = fl.Fluorophore(name="some_dye", position=[0, 0])
pprint(fluorophore)

In [ ]:
fluorophore = fl.Fluorophore(name="cy5_dna", position=[0, 0])
pprint(fluorophore)

A fluophore system contains  the fluorophore identities, their spatial arrangement and transition rate constants.

In [ ]:
fluorophores = [fluorophore]
fluorophore_system = fl.FluorophoreSystem(fluorophores=fluorophores)

In [ ]:
fluorophore_system.plot(scale=1, quadratic=True);

## Fluorophore data

The transition rate constants are specific for a kind of fluorophore and the chemical environment. 

Standard datasets with all constant photophysical attributes of the fluorophore are defined for various fluorophores.

In [ ]:
pprint(fluopy.fluo_data.atto643)

In [ ]:
pprint(fluopy.fluo_data.cy5_dna)

The data is collected in a dataclass specifying possible constants `fluopy.fluo_data.FluorophoreData`.

In [ ]:
fluorophore_data = fluopy.fluo_data.FluorophoreData()
pprint(fluorophore_data)

## Transition set

A set of possible transitions can be generated from a given fluorophore system by helper functions:

In [ ]:
transitions = fluorophore_system.load_transitions(
    summarize=False,
    irradiance=2.5,
    wavelength=640,
    bleaching=False,
    energy_transfer=False,
    dstorm=False,
)
pprint(transitions)

A transition set has to be defined from the individual transitions for a given fluorophore system:

In [ ]:
transition_set = tr.TransitionSet(transitions, fluorophore_system)

TransitionSet provides helper methods so that a transition_set can be easily modified with the following methods:

In [ ]:
transition_set_no_bleaching = transition_set.remove_absorbing_states()
transition_set_no_ets = transition_set.remove_energy_transfers()
transition_set_filtered = transition_set.filter_by_identity(remove_list=[6])

transition_set_adj = transition_set.adjust_rates(
    change_dict={1: 1e3}, keep_zero_rates=False
)

If for no other reasons you should remove transitions with rate constants of zero (this has possibly already been done above through keep_zero_rates=False):

In [ ]:
transtition_set = transition_set.remove_zero_rates()

A transition set has to be finalized before a simulation is started:

In [ ]:
transition_set.finalize()
list(vars(transition_set).keys())

A graphical representation of all possible transitions is given:

In [ ]:
transition_set.plot(graph_type="shell", colors=None, scale=1);

All transitions are listed in the `transitions`attribute or in a dataframe:

In [ ]:
transition_set.transition_df

Note: In transition_set.single_states, only the 'relevant' states are mentioned - the ones that actually occur in transition_set.plot().

Note: Transitions that have a rate of 0 or are not usable by the fluorophore do not occur transition_set.single_states; however, transitions that may not be reachable still do.

## Single dye simulation setup

For further demonstration we set up a simple fluorophore system that contains a single Cy5 fluorophore.

### Define the fluorophore system

In [ ]:
fluorophore = fl.Fluorophore(name="cy5_dna", position=[0, 0])
fluorophore_system = fl.FluorophoreSystem(fluorophores=[fluorophore])

### Define the transition set

In [ ]:
transitions = fluorophore_system.load_transitions(
    summarize=True,
    irradiance=2,
    wavelength=640,
    bleaching=False,
    energy_transfer=False,
    dstorm=False,
)
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set = transition_set.remove_energy_transfers()
transition_set.finalize()

## Make a prediction

For a fluorophore system and transition set that fullfill certain requirements a prediction for the simulation outcome can be derived based on computational estimation of a steady-state equilibrium.

In [ ]:
prediction = pr.Prediction(transition_set)
list(vars(prediction).keys())

The results are accessible through methods of the prediction instance.

In [ ]:
prediction.plot_frequency_transitions(scale=0.5)
prediction.plot_frequency_states(scale=0.5)
prediction.plot_mean_lifetimes(scale=0.5)
prediction.plot_mean_transition_times(scale=0.5)
prediction.plot_state_occupations(scale=0.5)
prediction.plot_transition_time_distributions(
    fluorophore="cy5_dna", transition_id=0, scale=0.5
);

Note for multi-fluorophore systems:
* if energy transfers occur, pairs have separate color in plot but are normalized with other transitions of their donor
* absorbing states: the prediction works using the fundamental matrix. For Q, the transitions leading to an OVERALL absorbed state are dropped. This can be done since bleaching is usually by far the least occurring transition, meaning each fluorophore will reach its limiting distribution (as if it was no absorbing Markov Chain)
        the frequencies and state occupations of absorbing transitions and states are set to 0, their lifetimes to inf (this is different to analysis, where - if absorbing transitions happened - only the lifetime and the state occupation is 0)



## Run a simulation

To carry out the complete Markov chain simulation, a Simulation object has to be instantiated from the transition set.

The simulation can then run for a given number of transition steps, or until it reaches a defined end time.

In [ ]:
%%time
simulation = si.Simulation(transition_set)
list(vars(simulation).keys())

In [ ]:
%%time
# simulate until it reaches given end_time
simulation.run(start_at=None, size=1e6, end_time=1, seed=rng, use_memmap=None)
mi.print_class(simulation)

If the simulation goes through many transtition events, the arrays may become too large for the available RAM.
In that case, use memory maps to store the data on the local drive.

## Run an approximate simulation

Under certain requirements (similar to prediction) a simulation approximation can can be carried out, saving time.

In [ ]:
%%time
approximation = si.Simulation(transition_set=transition_set)
approximation.approximate(prediction=prediction, size=1e6, seed=rng)
list(vars(approximation).keys())

## Analyze the simulation

Analyze the data from a given simulation for state probabilities and lifetimes, among others.

In [ ]:
analysis = an.Analysis(simulation=simulation)
list(vars(analysis).keys())

In [ ]:
mi.print_class(analysis)

In the following analysis plots the simulation results are overlaid with prediction outcomes (crosses) if a prediction is provided.

In [ ]:
analysis.get_fluorescence_lifetimes(fluorophore="cy5_dna")
analysis.get_emitting_transition_lifetimes(fluorophore="cy5_dna")

analysis.plot_frequency_transitions(scale=0.5, prediction=prediction)
analysis.plot_frequency_states(scale=0.5, prediction=prediction)
analysis.plot_mean_transition_times(scale=0.5, prediction=prediction)
analysis.plot_mean_lifetimes(scale=0.5, prediction=prediction)
analysis.plot_state_occupations(scale=0.5, prediction=prediction)
analysis.plot_lifetime_distributions(
    scale=0.5, prediction=prediction, fluorophore="cy5_dna", state_identity=1
)
analysis.plot_transition_time_distributions(
    scale=0.5, prediction=prediction, fluorophore="cy5_dna", transition_id=0
);

Note for multi-fluorophore systems:

* The analysis follows these rules:

    * time to transitions of energy transfers are collected only from the donor's point of view.

    * time to transition of e.g., fluorescence, does not differentiate whether energy transfer was also an option or not one energy transfer transition occurrence resembles the transition of the donor and the acceptor state lifetimes and occurrences do not differentiate whether energy transfer was involved or not.

* Energy transfer has an impact on true fluorescence lifetime only if the acceptors are truly available. If the potential acceptor is in a non-accepting state, the energy transfer is not available. If multiple fluorophores are potential acceptors, all of their respective rates are taken into account. Depending on the photophysical system, energy transfers may only have a marginal impact on the mean fluorescence lifetime.


## Simulation of experimentally observable (photons per frames) only

If you are not interested in all hidden state transitions but only in the observable photon emission events, you can limit the simulation to the relevant transitions.

This approach requires less memory than a full simulation. 

However, the complete statistical analysis is not available. A photon time series, FCS and blinking analysis is available.

The observation of emitted photons is further influenced by spectral filters, optical elements and detector noise contributions.

For multicolor fluorophore-systems, instantiate two Emissions objects with different bandpasses.

#### Extract photon emission events from simulation

In [ ]:
%%time
emissions = em.Emissions(frame_time="5ms", seed=rng, bandpass=(600, 800))
emissions.extract(simulation=simulation)
emissions

#### Simulate photon emission events

Helper routines are available to further modify the emission time series and correct for detection efficiency and noise contributions:

In [ ]:
emissions.add_photon_collection_objective(p=0.1, seed=rng)  # 1.
emissions.add_quantum_efficiency(p=0.9, seed=rng)  # 3.1.
emissions.add_poisson_noise(
    rate=0.05, seed=rng
)  # 3.2. (dark noise), note the frame time
emissions.add_emccd_gain(emccd_gain=10, seed=rng)  # 4.
emissions.add_gaussian_noise(mean=10, std=1, seed=rng)  # 5. (readout noise)

In [ ]:
emissions.plot_time_series();

In [ ]:
emissions.plot_histogram();

In [ ]:
emissions.plot_cumulative_events();

## Simulation of pulsed excitation

A simulation of observable photon emissions with pulsed laser excitation can be carried out. The analysis resembles a TSCPC experiment.

In this simulation, the returned lifetimes are the time differences of photon emission to the last laser pulse. I.e., if the time between laser pulses is very short, it may shorten the observed lifetimes. 

The simulation does not discriminate between multiple identical fluorophores (i.e., if one fluorophore gets excited, it transfers the energy to another fluorophore, and this fluorophore emits a photon). 

However, if the fluorophores are different, only one of them may be directly excitable by the laser pulse (due to the wavelength) and, depending on the provided bandpass, only the fluorescence of the other may be detected.

If details is True, a complete simulation object will be returned with transition_series, state_series and time_series (requiring more memory). 

In [ ]:
%%time
emissions_tcspc = em.Emissions(frame_time="10us", seed=rng, bandpass=(600, 800))
lifetimes_DA, lifetimes_D, lifetimes_all, simulation_object = emissions_tcspc.tcspc(
    transition_set=transition_set,
    number_pulses=1e5,
    pulse_duration=1e-11,
    time_between_pulses=1e-7,
    excitation_rates={"cy5_dna": 1e11},
    size=1e5,
    store_time_points=True,
    # details = True
)

In [ ]:
emissions_tcspc.plot_time_series()
fi.universal_figure(
    type_="hist", data=lifetimes_all, ylabel="PD", density=True, xlabel="Lifetime (s)"
);

Note:

Since the TCSPC simulation assumes a laser pulse to be instantaneous, a laser pulse that excited multiple fluorophores will lead to duplicates in the time series. This is handled by introducing minimal amount of spaces between these duplicates. The excitations are added in the correct order (fluorophore-specific) such that these series exactly represent the transition chain, even with multiple different distances between energy transfer partners. Note that when monitoring the fluorescence lifetime via the simulation object it will be the true one (as opposed to the generally returned lifetimes, which are the observed ones).

## Fluorescence correlation spectroscopy

Observed fluorescence emission events can be analyzed by a correlation analysis.

In [ ]:
fcs = fcs_p.FCS(emissions)
list(vars(fcs).keys())

### Autocorrelation of time points

The autocorrelation can be computed on the exact time points (being limited by the smallest available time difference).

In [ ]:
%%time
fcs.autocorrelate_time_points(
    exp_min=-16, exp_max=0, points_per_base=4, base=4, normalize=True
)

In [ ]:
mi.print_class(fcs)
fcs.plot(normalize_to=None, unit="s", scale=1);

### Autocorrelation of time series

The autocorrelation can be computed on the binned time series (being limited by the time bin).

There is not much to be seen in this example, since the bin time is larger than all relevant fluctuation time scales.

In [ ]:
fcs.autocorrelate_time_series(log=True, m=4, normalize=True)

In [ ]:
mi.print_class(fcs)
fcs.plot(normalize_to=None, unit="s", scale=1);

Some fcs fits are available:

In [ ]:
# fcs_predict = fcs_p.fit_dark(tau, dark_lifetime, dark_occupation)
# fcs_predict = fcs_p.fit_antibunching(tau, excitation_rate, s1_lifetime)
# fcs_predict = fcs_p.fit_triplet_cis(tau, k_isc, k_T, k_01, k_10, k_iso, k_biso_eff)

## Antibunching

Alternatively, you can focus on fast time scales in a linear scale and observe photon bunching or antibunching.

In [ ]:
# sensible to tau_max and bin_width, see coincidence notebook
hist, bins = fcs_p.coincidence(
    emissions.event_time_points[: int(2e5)], tau_max=1e-7, bin_width=1e-9, seed=rng
)
fi.universal_figure(
    type_="line",
    data=[bins, hist],
    xlabel=r"$\tau$ (s)",
    ylabel=r"$g^{(2)}(\tau)$",
    scale=1,
);

## Blinking

For analyzing a fluorescence on/off process, the blinking time scales can be estimated by thresholding an emission time series.

An ON-period is a number of consecutive frames where each frame contains a minimum amount of emissions (> threshold). 

An OFF-period is a number of consecutive frames where each frame contains a maximum amount of emissions (≤ threshold). 

Each ON-period is followed by an OFF-period and vice versa.

### Emissions from a short simulation

We limit the dataset to 2000 frames for illustration purposes.

In [ ]:
%%time
emissions = em.Emissions(frame_time="10us", seed=rng, bandpass=None)
emissions.simulate(transition_set=transition_set, store_time_points=False, frames=2000)
emissions

In [ ]:
threshold: int = 5

In [ ]:
emissions.plot_time_series(scale=1)
plt.hlines(threshold, xmin=0, xmax=0.02)

In [ ]:
blinks = bl.Blinking(emissions, threshold=threshold)
blinks

In [ ]:
mi.print_class(blinks)

In [ ]:
# plot a histogram of OFF times
blinks.plot(
    mode="off_histogram", density=True, display_mean=True, as_time="ms", scale=0.5
)

# plot a histogram of ON times
blinks.plot(
    mode="on_histogram", density=True, display_mean=True, as_time="ms", scale=0.5
)

# plot a time series of OFF times
blinks.plot(mode="off_frame_series", scale=0.5)

# plot a time series of ON times
blinks.plot(mode="on_frame_series", scale=0.5);

You can extract the analytical OFF statistics from the emission time series without differentiating between fluorophores.

In [ ]:
on_off_times_analytic, on_off_values_analytic = bl.get_analytical_off_statistics(
    off_frames=blinks.off_periods_frames,
    off_periods=blinks.off_periods,
    on_frames=blinks.on_periods_frames,
    frame_time=blinks.emissions.parameters["frame_time"],
)

bl.plot_off_statistics(
    on_off_times_analytic, on_off_values_analytic, scale=1, title="analytical OFF"
);

### Emissions from the long simulation

Get more detailed information from a complete simulation:

In [ ]:
%%time
emissions = em.Emissions(frame_time="200ns", seed=rng, bandpass=None)
emissions.extract(simulation=simulation)
emissions

In [ ]:
blinks = bl.Blinking(emissions, threshold=threshold)
blinks

In [ ]:
mi.print_class(blinks)

In [ ]:
# plot a histogram of OFF times
blinks.plot(
    mode="off_histogram", density=True, display_mean=True, as_time="ms", scale=0.5
)

# plot a histogram of ON times
blinks.plot(
    mode="on_histogram", density=True, display_mean=True, as_time="ms", scale=0.5
)

# plot a time series of OFF times
blinks.plot(mode="off_frame_series", scale=0.5)

# plot a time series of ON times
blinks.plot(mode="on_frame_series", scale=0.5)